# This notebook describes the workflow for quality control and ingesting the data into a set of hd5 files per person

## Set up your environment
You will want to set your Cloud compute profile to have: 16 CPUs, 104GB RAM, 1000GB Disk (note that this is at the bottom) and the Compute type should be a Dataproc Cluster You'll note that there's a handy link nearby that you can use to monitor your usage. We like to use the Yarn dashboard.

You will want to pick 4 Workers and 8 Preemptible workers with the following config 16 CPUs, 104 GB RAM and 1000GB Disk

[This setup lets you run upto 7 notebooks parallely (so you can ingest 7000 particpant data in parallel]

Go ahead and restart your environment!

## Setup spark

There's a lot of fitbit data to ingest, so we are going to use Dataproc and Spark to parallelize and process faster

In [ ]:
## Install pyspark! If you get errors, ensure that the version matches with the notebook python version
%pip install pyspark

In [ ]:
## Import spark libraries
from pyspark.context import SparkContext
from pyspark.sql.session import SparkSession
from pyspark.sql.functions import col, collect_list, to_json, struct, to_date, split, date_format
from pyspark.sql import functions as F

In [ ]:
## Import other libraries
import os

os.environ["WORKSPACE_CDR"] = #specify Google Project
os.environ["WORKSPACE_BUCKET"] = #specify GCP bucket


import pandas as pd
import time
import h5py
import json
import numpy as np
import gcsfs
from datetime import datetime, timedelta
import subprocess
from typing import Dict


In [ ]:
## Set up your spark session -- feel free to name it as you would like. 
## We chose 'spark-bigquery-ingest_data_cmp_ehr' which is how it will show up in the Yarn dashboard
spark = SparkSession \
  .builder \
  .master('yarn') \
  .appName('spark-bigquery-ingest_data_cmp_ehr') \
  .getOrCreate()

## load data as spark dataframes

In [ ]:
## Get the workspace project info so that we can use this when pulling data out of BigQuery
bucket_= # specify local fuse mounted path
BUCKET = # specify gs bucket path


#paths that are hardcoded to save files generated
file_paths_values = pd.read_csv('filepaths_values.csv') # load your version(loading the exmaple version share here, you can renmae based on your preferences)

file_paths_values = file_paths_values[file_paths_values['notebook']=='qc_ingestion'][['notebook','key_','value_']]
paths_dict = file_paths_values.set_index("key_")["value_"]

In [ ]:
no = 1#change depending on chunk you are processing
folder_type = paths_dict.get("ehr_save")#name of the folder to grab person_ids from
folder_to_save = paths_dict.get("folder_ingest_ehr")#name of folder to save hd5 files
folder_sleep = paths_dict.get("sleep_import")#name of the folder to grab processed sleep data from

In [ ]:
# read corresponding chunk's HR data as a spark dataframe
df = (spark.read.format("csv")
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .load(f'{bucket_}/{folder_type}/chunk_{no}/fitbit_heart_rate_level_second_chunk_{no}.tsv'))


In [ ]:
# read corresponding chunk's steps data as a spark dataframe
df2 = (spark.read.format("csv")
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .load(f'{bucket_}/{folder_type}/chunk_{no}/fitbit_intraday_updated_steps_chunk_{no}.tsv'))

In [ ]:
# read corresponding chunk's processed and overlap fixed sleep data as a spark dataframe
df3 = (spark.read.format("csv")
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .load(f'{bucket_}/{folder_sleep}/overlapped_entries_df_chunk_{no}.tsv'))

df3 = df3.withColumn("date", split(col("start_datetime"), " ").getItem(0))


In [ ]:
#aggregate all sleep stages at a day level
df_inner_agg = (df3.groupBy("person_id", "date")
                .agg(
                    F.collect_list(
                        F.struct("start_datetime","end_datetime", "is_main_sleep", "duration_in_min", "level")
                    ).alias("sleep_records_for_date")
                )
               )
#aggregate all sleep stages at a day level to a per person level
df_outer_agg = (df_inner_agg.groupBy("person_id")
                .agg(
                    F.collect_list(
                        F.struct("date", "sleep_records_for_date")
                    ).alias("sleep_by_date")
                )
                .withColumn("sleep_json", F.to_json("sleep_by_date"))
               )

In [ ]:
# Join the other day level data
df = df.join(df2, on=['date','person_id'], how='outer')

In [ ]:
#aggregate other day level data to a per person level
result_df = (df.groupBy("person_id")
             .agg(
                 collect_list(
                     struct("date", "src_id", "sec_timestamps_str", "sec_heart_rate_array_str", "raw_data_count", "heart_rate_array_str", "non_zero_count", "steps_array_str", "non_zero_step_count")
                ).alias("heartrate_and_steps")
             )
             .withColumn("heartrate_and_steps_json", to_json("heartrate_and_steps"))
            )

In [ ]:
#join sleep per person level data to other per person level
result_df_ = df_outer_agg.join(result_df, on=['person_id'], how='outer')

## Quality control functions and constants
### Rules to QC the data
A large part of dealing with any data is understanding how to filter your data. We have decided on several rules which are outlined below. This will ultimately improve our ML models.

**Finalized QC thresholds:**

1. We are replacing unfeasible heart-rate values lower than 30 and higher than 200 with nan
1. We are also replacing unfeasible step count per minute, so the sum of steps do not turn out to be in millions like we had observed in some cases.  We checked the distribution of step counts per minute to find the threshold for the 99th percentile, it was 101, but since during marathon cadence can go to 180-200, we will keep the threshold as 250, and replace anything above that with nan.
1. For flagging a data to be considered or not, we looked at the distribution of wear-time minutes. And found 200 to have a percentile of 1. So we will flag any arrays that have less than 200 minutes of data.  


### Minute array construction rules
Aggregate sleep segment data into per-minute arrays and summary statistics  


Ends at XX:30 → extend to XX+1:00.
Previous ended at XX:30 → next starts at XX+1:00.
Store in HDF5 (compact int mapping)  

We are also saving interpolated heartrate and steps array with sleep time removed, currently we have details for sleep as being in {wake, rem, deep, light, asleep, awake, restless} modes. We will only keep wake or awake time samples, but remove wake, awake time in between sleep.

### QC functions

In [ ]:
#constants and functions for 
steps_threshold = 250
hr_low = 30
hr_high = 200
missingness_threshold = 200


# String stage -> compact integer code (for storage)
SLEEP_STAGE_MAPPING: Dict[str, float] = {
    "none": np.nan,
    "deep": 0,
    "wake": 1,
    "light": 2,
    "rem": 3,
    "asleep": 4,
    "restless": 5,
    "awake": 6,
    "unknown": 7,
    # non-main variants
    "nonmain_deep": 8,
    "nonmain_wake": 9,  
    "nonmain_light": 10,
    "nonmain_rem": 11,
    "nonmain_asleep": 12,
    "nonmain_restless": 13,
    "nonmain_awake": 14,
    "nonmain_unknown": 15,
}

# Reverse map for decoding arrays
INT_TO_SLEEP_STAGE: Dict[float, str] = {v: k for k, v in SLEEP_STAGE_MAPPING.items()}


    


# -----------------------------------------------------------------------------
# helper functions
# -----------------------------------------------------------------------------

# Minute array construction with half-minute rules
def create_minute_array_with_precision(segments_df: pd.DataFrame) -> np.ndarray:
    """
    Create a minute-by-minute array with special handling for half-minute transitions:
    - First segment fills normally
    - If a segment ends at XX:30, it fills until XX+1:00
    - Next segment starts at XX+1:00

    Args:
        segments_df: DataFrame with start_datetime, end_datetime, level columns
    Returns:
        NumPy array with minute-by-minute sleep stages
    """
    minute_array = np.zeros(1440, dtype='U20')
    minute_array.fill('none')
    segments_df = segments_df[segments_df['duration_in_min']!=0]
    segments = segments_df.copy().sort_values('start_datetime')

    prev_end_minute = None

    for i, row in segments.iterrows():
        start = row['start_datetime']

        if 'end_datetime' in row and pd.notna(row['end_datetime']):
            end = row['end_datetime']
        else:
            end = start + pd.Timedelta(minutes=row['duration_in_min'])

        if not row['is_main_sleep']:
            add = 'nonmain_'
        else:
            add = ''
        level = add+row['level']

        start_minute_exact = start.hour * 60 + start.minute + start.second/60
        end_minute_exact = end.hour * 60 + end.minute + end.second/60

        # Extract whole minutes and seconds for rule processing
        start_minute = start.hour * 60 + start.minute
        start_second = start.second
        end_minute = end.hour * 60 + end.minute
        end_second = end.second

        # Apply special rules for segments after the first one
        if i > 0 and prev_end_minute is not None and (prev_end_minute.hour!=23 and prev_end_minute.minute!=59):
            # If previous segment ended at XX:30, start this one at XX+1:00
            if prev_end_minute.second == 30:
                rounded_prev_end = prev_end_minute.replace(second=0) + pd.Timedelta(minutes=1)
                # Adjust start time if it's earlier than the rounded end of previous segment
                if start < rounded_prev_end:
                    start = rounded_prev_end
                    start_minute = start.hour * 60 + start.minute
                    start_second = 0

        # Fill the minute array
        start_idx = start_minute

        # If segment ends at XX:30, extend to fill until XX+1:00
        # This applies to ALL segments including the last one
        if end.hour == 0 and end.minute == 0 and end.second == 0:
        # If segment ends at exactly midnight, it should fill until 23:59
            end_idx = 1440 
        else:
            if end_second == 30:
                end_idx = end_minute + 1  # Fill until next minute
            else:
                # Round to nearest minute
                end_idx = end_minute
                if end_second > 0:
                    end_idx += 1

#             Ensure bounds
        start_idx = max(0, min(start_idx, 1439))
        end_idx = max(0, min(end_idx, 1440))

        # Fill the array
        minute_array[start_idx:end_idx] = level

        # Store the end time for next iteration
        prev_end_minute = end

    return minute_array


def array_with_fixed_mapping(
    string_array: np.ndarray
) -> None:
    """Write a string sleep-array to HDF5 after mapping to small int codes.

    Parameters
    ----------
    group : h5py.Group
        Open HDF5 group to receive the dataset.
    name : str
        Dataset name (e.g., 'sleep_array').
    string_array : np.ndarray
        Array of string stages (length 1440 for one day).
    compression : str, optional
        HDF5 compression filter (default 'gzip').
    """
    int_array = np.zeros(len(string_array), dtype=np.float64)

    # Map each label to its integer code; default to 88 ('none') if missing
    for i, val in enumerate(string_array):
        int_array[i] = SLEEP_STAGE_MAPPING.get(val, 88)

    return int_array
    

In [ ]:
#we will store data in three seperate files per person
raw_directory = f'chunk_{no}_raw'
qc_directory = f'chunk_{no}_qc'
os.makedirs(raw_directory, exist_ok=True)
print(f"Directory '{raw_directory}' is ready!")
os.makedirs(qc_directory, exist_ok=True)
print(f"Directory '{qc_directory}' is ready!")


### Ingestion with QCing

In [ ]:
###data writing with qc

start = time.time()

# Process each person's data from the DataFrame
for row in result_df_.select("person_id", "heartrate_and_steps_json","sleep_json").toLocalIterator():
    person_id = row['person_id']
    heartrate_json = row['heartrate_and_steps_json']
    sleep_json = row['sleep_json']
    
    # Parse JSON data to Python objects, or use empty lists if None
    heartrate_data = json.loads(heartrate_json) if heartrate_json is not None else []
    sleep_data = json.loads(sleep_json) if sleep_json is not None else []
     # Create a dictionary mapping date strings to sleep records for quick lookup
    sleep_data_by_date = {record['date'][0:10] + ' 00:00:00': record for record in sleep_data}

    # Open three HDF5 files for each person: raw data, quality controlled data, and sleep data
    with h5py.File(f'chunk_{no}_raw/{person_id}.hd5', 'w') as hdf_raw, \
    h5py.File(f'chunk_{no}_qc/{person_id}.hd5', 'w') as hdf_qc: 
        # Process each heart rate and steps record
        for record in heartrate_data:
            date_str = record['date'][0:10] + ' 00:00:00'  # Format date string
            
            # Create or get groups for this date in each HDF5 file
            raw_date_group = hdf_raw.require_group(date_str)
            qc_date_group = hdf_qc.require_group(date_str)
            
            src_id = str(record['src_id']) if 'src_id' in record and str(record['src_id']) is not None else 'PTSC' 
            
            
            # Process heart rate data if it exists
            if 'heart_rate_array_str' in record and record['heart_rate_array_str'] is not None:
                heart_rate = np.array([float(x) for x in record['heart_rate_array_str'].split(',')])
                if 'sec_heart_rate_array_str' in record and record['sec_heart_rate_array_str'] is not None:
                    heart_rate_sec = np.array([float(x) for x in record['sec_heart_rate_array_str'].split(',')])
                    hr_timestamps_sec = np.array([str(x) for x in record['sec_timestamps_str'].split(',')], dtype=h5py.string_dtype()) 
                    hr_sec_info = float(record['raw_data_count']) 
                else:
                    heart_rate_sec = None
                
                heart_rate[heart_rate == 0] = np.nan##QC: where fitbit data is missing, 0--np.nan
                heart_rate_p = heart_rate.copy()
                bad_idx_hr = np.where((heart_rate_p > hr_high) | (heart_rate_p < hr_low))[0]
                heart_rate_p[(heart_rate_p > hr_high) | (heart_rate_p < hr_low)] = np.nan ##QC: erraneous values
                
                


            else:
                heart_rate = np.array([])
                heart_rate_p = np.array([])
                heart_rate_sec = None
                bad_idx_hr = np.array([])
                

            if 'steps_array_str' in record and record['steps_array_str'] is not None:
                steps = np.array([float(x) for x in record['steps_array_str'].split(',')])
                steps[(steps == -1)] = np.nan ##QC: where fitbit data is missing, -1--np.nan
                steps_p = steps.copy()
                bad_idx_st = np.where((steps_p > steps_threshold))[0]
                steps_p[(steps_p > steps_threshold)] = np.nan ##QC: erraneous values
                    
            else:
                steps = np.array([])
                steps_p = np.array([])
                bad_idx_st = np.array([])


            
            # Check if there's too much missing heart rate data after QC and interpolation
            if len(heart_rate_p) == 0 or \
                np.sum(np.isnan(heart_rate_p))>= missingness_threshold:
                missingness_flag_hr = True
            else:
                missingness_flag_hr = False

            missingness_info_hr = f"{np.sum(np.isnan(heart_rate_p))}/{len(heart_rate_p)}" 

            
            if len(steps_p) ==0 or \
                np.sum(np.isnan(steps_p))>= missingness_threshold:
                missingness_flag_steps = True
            else:
                missingness_flag_steps = False

            missingness_info_steps = f"{np.sum(np.isnan(steps_p))}/{len(steps_p)}" 

            
            # Process sleep data - get records from current day
            sleep_record_t = sleep_data_by_date.get(date_str)
            
            
            if sleep_record_t:
                sleep_record = (sleep_record_t.get('sleep_records_for_date', []) if sleep_record_t else []) 
                
                
                
                # Convert sleep records to minute-by-minute representation
                if sleep_record:
                    sleep_df_record = pd.DataFrame(sleep_record)
                    sleep_df_record["start_datetime"] = pd.to_datetime(sleep_df_record["start_datetime"],format='ISO8601')
                    sleep_df_record["end_datetime"] = pd.to_datetime(sleep_df_record["end_datetime"],format='ISO8601')

                    sleep_str_array = create_minute_array_with_precision(sleep_df_record)
                    sleep_type_distribution = array_with_fixed_mapping(sleep_str_array)

                    
                else:
                    sleep_type_distribution = np.array([]) 
            
            else:
                sleep_type_distribution = np.array([]) 
                
            
            
            
            
            
            # Store raw heart rate data in HDF5
            raw_date_group.create_dataset('raw_hr_array', data=heart_rate,compression="gzip")
            raw_date_group.create_dataset('nullified_hr_indices', data=bad_idx_hr,compression="gzip")
            
            # Store QC'ed and interpolated heart rate data
            qc_date_group.create_dataset('hr_array_qc', data=heart_rate_p,compression="gzip")
            
            # Store raw second level heart rate arrays if available
            if heart_rate_sec is not None:
                raw_date_group.create_dataset('raw_hr_array_seconds', data=heart_rate_sec,compression="gzip")
                raw_date_group.create_dataset('raw_hr_available_timestamps_seconds', data=hr_timestamps_sec,compression="gzip")
                raw_date_group.create_dataset('no_of_seconds_of_hr_data', data=hr_sec_info)
                raw_date_group.create_dataset('hr_seconds_array_flag', data=True)
            
            else:
                raw_date_group.create_dataset('hr_seconds_array_flag', data=False)
                
            # Store heart rate QC flags and info
            qc_date_group.create_dataset('hr_missingness_flag', data=missingness_flag_hr)
            qc_date_group.create_dataset('hr_missingness_info', data=missingness_info_hr)
            
#             # Store sleep data
            qc_date_group.create_dataset('sleep_array', data=sleep_type_distribution,compression="gzip")

            
            # Store raw steps data in HDF5
            raw_date_group.create_dataset('steps_array', data=steps,compression="gzip")
            raw_date_group.create_dataset('nullified_steps_indices', data=bad_idx_st,compression="gzip")
            # Store QC'ed and interpolated steps data
            qc_date_group.create_dataset('steps_array_qc', data=steps_p,compression="gzip")
            # Store steps QC flags and info
            qc_date_group.create_dataset('steps_missingness_flag', data=missingness_flag_steps)
            qc_date_group.create_dataset('steps_missingness_info', data=missingness_info_steps)
            

    
end = time.time()
print(f"Process completed in {end-start} seconds.")

### Save data to bucket

In [ ]:
# Run the GCS copy command to copy data to bucket
subprocess.run(f"gcloud storage cp -r chunk_{no}_raw/ {bucket_}/{folder_to_save}/", shell=True, check=True)
subprocess.run(f"gcloud storage cp -r chunk_{no}_qc/ {bucket_}/{folder_to_save}/", shell=True, check=True)


In [ ]:
# remove data locally if needed
subprocess.run(f"rm -r chunk_{no}_raw", shell=True, check=True)
subprocess.run(f"rm -r chunk_{no}_qc", shell=True, check=True)


# Follow along in the '003_trends_calculation' notebook for next steps
There we do temporal trend calculations for activity levels from the ingested data